# 🧬 NonBScanner - Quick Analysis Workflow

**Complete workflow for Non-B DNA motif detection and analysis**

**100X PERFORMANCE MODE**: All 9 detectors run in parallel using ThreadPoolExecutor

This notebook provides a streamlined approach to:
1. **Setup & Analysis**: Import libraries and analyze FASTA sequences (with progress tracking)
2. **Results & Export**: View results and export to Excel/CSV
3. **Visualization**: Generate comprehensive plots and statistics

---

## Box 1: Setup and Analysis

### Import Libraries and Analyze Sequences (100X Performance Mode)

In [ ]:
# ============================================================================
# BOX 1: SETUP AND ANALYSIS (100X PERFORMANCE MODE)
# ============================================================================

# Import NonBScanner modules
from nonbscanner import analyze_sequence, analyze_multiple_sequences, get_motif_info, CHUNK_THRESHOLD, DEFAULT_CHUNK_SIZE
from utilities import (
    read_fasta_file, export_to_excel, export_to_csv, export_to_json,
    export_results_to_dataframe, get_basic_stats
)
from visualizations import (
    plot_motif_distribution, plot_coverage_map, plot_nested_pie_chart,
    plot_length_distribution, plot_score_distribution
)
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from datetime import datetime
import time

print("✅ Libraries imported successfully!")
print(f"\n🚀 100X PERFORMANCE MODE: 9 detectors run in parallel")
print(f"📊 Chunk threshold: {CHUNK_THRESHOLD:,} bp (sequences larger than this are processed in chunks)")
print(f"📊 Default chunk size: {DEFAULT_CHUNK_SIZE:,} bp")

print("\n📋 Available Motif Classes:")
motif_info = get_motif_info()
print(f"Total Classes: {motif_info.get('total_classes', 0)}")
classification = motif_info.get('classification', {})
for cls_id, info in list(classification.items())[:5]:
    print(f"  - {info['name']}: {', '.join(info['subclasses'][:2])}...")
print(f"  ... and {len(classification) - 5} more!\n")

# ============================================================================
# ANALYZE YOUR FASTA FILE WITH PROGRESS TRACKING
# ============================================================================

# Specify your FASTA file path
fasta_file = "example_motifs_multiline.fasta"  # Change this to your file

# Parse FASTA file
sequences = read_fasta_file(fasta_file)
print(f"📄 Loaded {len(sequences)} sequence(s) from {fasta_file}")

# Calculate total size
total_bp = sum(len(seq) for seq in sequences.values())
print(f"📊 Total sequence data: {total_bp:,} bp")

# Progress tracking for large sequences
def progress_callback(current_chunk, total_chunks, bp_processed):
    """Callback for chunk-level progress tracking."""
    pct = (current_chunk / total_chunks) * 100 if total_chunks > 0 else 100
    print(f"   └─ Chunk {current_chunk}/{total_chunks} ({pct:.1f}%) - {bp_processed:,} bp processed", end='\r')

# Analyze all sequences with timing and progress
all_results = []
start_time = time.time()

for i, (seq_name, seq) in enumerate(sequences.items(), 1):
    seq_len = len(seq)
    is_large = seq_len > CHUNK_THRESHOLD
    
    print(f"\n🔬 [{i}/{len(sequences)}] Analyzing: {seq_name} ({seq_len:,} bp){'  [CHUNKED]' if is_large else ''}")
    
    seq_start = time.time()
    
    # Use progress callback for large sequences
    if is_large:
        results = analyze_sequence(seq, seq_name, progress_callback=progress_callback)
        print()  # New line after progress
    else:
        results = analyze_sequence(seq, seq_name)
    
    seq_time = time.time() - seq_start
    throughput = seq_len / seq_time if seq_time > 0 else 0
    
    all_results.append({
        'name': seq_name,
        'sequence': seq,
        'motifs': results
    })
    print(f"   ✓ Found {len(results)} motifs in {seq_time:.2f}s ({throughput:,.0f} bp/s)")

total_time = time.time() - start_time

# Combine all motifs
all_motifs = []
for result in all_results:
    for motif in result['motifs']:
        if 'Sequence_Name' not in motif:
            motif['Sequence_Name'] = result['name']
        all_motifs.append(motif)

# Performance summary
print("\n" + "=" * 60)
print("✅ ANALYSIS COMPLETE (100X PERFORMANCE MODE)")
print("=" * 60)
print(f"📊 Total motifs detected: {len(all_motifs)}")
print(f"🧬 Sequences analyzed: {len(all_results)}")
print(f"⏱️ Total time: {total_time:.2f} seconds")
print(f"🚀 Overall throughput: {total_bp / total_time:,.0f} bp/s")

---

## Box 2: Results and Export

### View Results, Summary Statistics, and Export Data

In [ ]:
# ============================================================================
# BOX 2: RESULTS AND EXPORT
# ============================================================================

# Convert results to DataFrame for viewing
df_results = export_results_to_dataframe(all_motifs)

print("📊 RESULTS SUMMARY")
print("=" * 80)

# Display summary by class
class_counts = df_results['Class'].value_counts()
print("\n📈 Motifs by Class:")
for cls, count in class_counts.items():
    print(f"  {cls:30s}: {count:4d} motifs")

# Display summary statistics
print("\n📏 Summary Statistics:")
print(f"  Total Motifs: {len(all_motifs)}")
print(f"  Unique Classes: {df_results['Class'].nunique()}")
print(f"  Unique Subclasses: {df_results['Subclass'].nunique()}")
print(f"  Average Length: {df_results['Length'].mean():.1f} bp")
print(f"  Average Score: {df_results['Score'].mean():.3f}")
print(f"  Length Range: {df_results['Length'].min()}-{df_results['Length'].max()} bp")

# Display first few rows
print("\n📋 First 10 Motifs:")
display(df_results[['Sequence_Name', 'Class', 'Subclass', 'Start', 'End', 'Length', 'Score']].head(10))

# ============================================================================
# EXPORT RESULTS
# ============================================================================

print("\n" + "=" * 80)
print("💾 EXPORTING RESULTS")
print("=" * 80)

# Export to Excel (multi-sheet with class breakdown)
excel_file = "nbdscanner_results.xlsx"
export_to_excel(all_motifs, excel_file)
print(f"✅ Excel file saved: {excel_file}")
print("   (Contains multiple sheets: Consolidated + per-class breakdown)")

# Export to CSV
csv_file = "nbdscanner_results.csv"
csv_data = export_to_csv(all_motifs)
with open(csv_file, 'w') as f:
    f.write(csv_data)
print(f"✅ CSV file saved: {csv_file}")

# Export to JSON
json_file = "nbdscanner_results.json"
json_data = export_to_json(all_motifs, pretty=True)
with open(json_file, 'w') as f:
    f.write(json_data)
print(f"✅ JSON file saved: {json_file}")

print("\n🎉 All exports complete!")

---

## Box 3: Comprehensive Visualization

### Generate Publication-Quality Plots and Analysis

In [ ]:
# ============================================================================
# BOX 3: VISUALIZATION AND ANALYSIS
# ============================================================================

import matplotlib.pyplot as plt
import seaborn as sns

# Use a compatible style that works across matplotlib versions
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    try:
        plt.style.use('seaborn-darkgrid')
    except:
        sns.set_style('darkgrid')

print("📊 GENERATING VISUALIZATIONS")
print("=" * 80)

# Set up figure layout
fig = plt.figure(figsize=(20, 12))

# ============================================================================
# 1. MOTIF CLASS DISTRIBUTION
# ============================================================================
print("\n1️⃣ Creating Motif Class Distribution...")
ax1 = plt.subplot(2, 3, 1)
fig1 = plot_motif_distribution(all_motifs, by='Class', 
                                title='Motif Class Distribution')
plt.close(fig1)  # We'll recreate in our main figure

class_counts = Counter([m.get('Class', 'Unknown') for m in all_motifs])
classes = list(class_counts.keys())
counts = list(class_counts.values())
ax1.bar(range(len(classes)), counts, color='steelblue', alpha=0.7)
ax1.set_xticks(range(len(classes)))
ax1.set_xticklabels(classes, rotation=45, ha='right')
ax1.set_xlabel('Motif Class')
ax1.set_ylabel('Count')
ax1.set_title('Motif Class Distribution', fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# ============================================================================
# 2. SUBCLASS DISTRIBUTION (TOP 10)
# ============================================================================
print("2️⃣ Creating Subclass Distribution...")
ax2 = plt.subplot(2, 3, 2)
subclass_counts = Counter([m.get('Subclass', 'Unknown') for m in all_motifs])
top_subclasses = dict(subclass_counts.most_common(10))
ax2.barh(list(top_subclasses.keys()), list(top_subclasses.values()), 
         color='coral', alpha=0.7)
ax2.set_xlabel('Count')
ax2.set_title('Top 10 Subclasses', fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

# ============================================================================
# 3. LENGTH DISTRIBUTION
# ============================================================================
print("3️⃣ Creating Length Distribution...")
ax3 = plt.subplot(2, 3, 3)
lengths = [m.get('Length', 0) for m in all_motifs]
ax3.hist(lengths, bins=30, color='green', alpha=0.7, edgecolor='black')
ax3.set_xlabel('Length (bp)')
ax3.set_ylabel('Frequency')
ax3.set_title('Motif Length Distribution', fontweight='bold')
ax3.grid(axis='y', alpha=0.3)

# ============================================================================
# 4. SCORE DISTRIBUTION
# ============================================================================
print("4️⃣ Creating Score Distribution...")
ax4 = plt.subplot(2, 3, 4)
scores = [m.get('Score', 0) for m in all_motifs]
ax4.hist(scores, bins=30, color='purple', alpha=0.7, edgecolor='black')
ax4.set_xlabel('Score')
ax4.set_ylabel('Frequency')
ax4.set_title('Motif Score Distribution', fontweight='bold')
ax4.grid(axis='y', alpha=0.3)

# ============================================================================
# 5. CLASS PIE CHART
# ============================================================================
print("5️⃣ Creating Class Pie Chart...")
ax5 = plt.subplot(2, 3, 5)
ax5.pie(counts, labels=classes, autopct='%1.1f%%', startangle=90)
ax5.set_title('Motif Class Proportions', fontweight='bold')

# ============================================================================
# 6. COVERAGE MAP (First sequence)
# ============================================================================
print("6️⃣ Creating Coverage Map...")
ax6 = plt.subplot(2, 3, 6)
if all_results:
    first_seq_motifs = all_results[0]['motifs']
    seq_length = len(all_results[0]['sequence'])
    
    for i, motif in enumerate(first_seq_motifs[:50]):  # Limit to 50 for visibility
        start = motif.get('Start', 0)
        end = motif.get('End', 0)
        ax6.plot([start, end], [i, i], linewidth=2, alpha=0.7)
    
    ax6.set_xlabel('Position (bp)')
    ax6.set_ylabel('Motif Index')
    ax6.set_title(f'Motif Coverage Map\n{all_results[0]["name"]}', fontweight='bold')
    ax6.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('nbdscanner_visualizations.png', dpi=300, bbox_inches='tight')
print("\n✅ Visualization saved: nbdscanner_visualizations.png")
plt.show()

# ============================================================================
# SUMMARY TABLE
# ============================================================================
print("\n" + "=" * 80)
print("📋 DETAILED SUMMARY TABLE")
print("=" * 80)

summary_data = []
for cls in sorted(class_counts.keys()):
    cls_motifs = [m for m in all_motifs if m.get('Class') == cls]
    if cls_motifs:
        summary_data.append({
            'Class': cls,
            'Count': len(cls_motifs),
            'Avg Length': f"{np.mean([m.get('Length', 0) for m in cls_motifs]):.1f}",
            'Avg Score': f"{np.mean([m.get('Score', 0) for m in cls_motifs]):.3f}",
            'Min Length': min([m.get('Length', 0) for m in cls_motifs]),
            'Max Length': max([m.get('Length', 0) for m in cls_motifs])
        })

summary_df = pd.DataFrame(summary_data)
display(summary_df)

print("\n" + "=" * 80)
print("🎉 ANALYSIS COMPLETE!")
print("=" * 80)
print(f"\n📊 Total Motifs: {len(all_motifs)}")
print(f"🧬 Sequences Analyzed: {len(all_results)}")
print(f"📁 Files Generated:")
print(f"   - nbdscanner_results.xlsx (Excel)")
print(f"   - nbdscanner_results.csv (CSV)")
print(f"   - nbdscanner_results.json (JSON)")
print(f"   - nbdscanner_visualizations.png (Plots)")
print(f"\n✨ All done! Your results are ready for publication.")